# 07장 보안 실습 — 계정·권한 기준선 검토


## Goal

UID 0·특수 비트·쓰기 권한을 정상 기준선과 비교합니다.

[교안과 분석 질문](../../07-secure-scripting/07-3-account-permission-review.md)을 먼저 읽습니다.


## Setup

Python 커널의 %%bash를 사용합니다. 새 임시 폴더에 합성 자료와 결과 경로를 준비합니다. 외부 접속·서비스 등록·원본 서버 조사는 하지 않습니다. 코드를 검토하고 Setup부터 순서대로 실행합니다. Bash 셀 사이의 상태는 환경 변수와 파일로 전달합니다.


In [ ]:
from pathlib import Path
import hashlib
import os
import tempfile

lab = Path(tempfile.mkdtemp(prefix='bash-security-07-'))
data = lab / 'data'
output = lab / 'output'
data.mkdir()
output.mkdir()
fixtures = {'passwd.sample': 'root:x:0:0:root:/root:/bin/bash\nanalyst:x:1000:1000:Analyst:/home/analyst:/bin/bash\ncollector:x:995:995:Collector:/var/lib/collector:/usr/sbin/nologin\nlegacy-admin:x:0:0:Legacy:/var/lib/legacy:/usr/sbin/nologin\n', 'permissions.psv': 'path|owner|group|mode|purpose|approval\n/usr/bin/passwd|root|root|4755|password-management|baseline\n/opt/collector/bin/report|root|collector|0775|service-executable|review\n/var/tmp/course-cache|root|root|1777|shared-temp|baseline\n'}
for name, content in fixtures.items():
    path = data / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
before = {name: hashlib.sha256((data / name).read_bytes()).hexdigest() for name in fixtures}
tools_dir = lab / 'tools'
tools_dir.mkdir()
os.environ['COURSE_TOOLS'] = str(tools_dir)
os.environ['COURSE_DATA'] = str(data)
os.environ['COURSE_OUT'] = str(output)
print('합성 자료와 새 결과 폴더 준비 완료')


## Steps

예상 결과: UID 0 계정 2개, 기준선 SUID 1개, 검토 항목 1개, 악용 입증 아님

명령을 실행하기 전에 입력·출력·실패 조건을 표시합니다. 자료의 상세 필드 해석과 정상 행위 대안은 연결된 교안에서 확인합니다.


### 1. UID 0 계정과 로그인 셸 구분


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
awk -F: '$3==0 {print $1 "|" $7}' "$COURSE_DATA/passwd.sample" > "$COURSE_OUT/uid0.psv"
test "$(wc -l < "$COURSE_OUT/uid0.psv")" -eq 2
grep -Fx 'legacy-admin|/usr/sbin/nologin' "$COURSE_OUT/uid0.psv"


### 2. 비트와 승인 기준선 함께 읽기


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
awk -F '|' 'NR>1 && $4 ~ /^[4567]/ {print $1 "|" $4 "|" $6}' \
 "$COURSE_DATA/permissions.psv" > "$COURSE_OUT/suid-review.psv"
grep -Fx '/usr/bin/passwd|4755|baseline' "$COURSE_OUT/suid-review.psv"


### 3. 검토 대상과 취약점 확정 구분


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
awk -F '|' 'NR>1 && $6=="review" {print $1 "|" $3 "|" $4}' \
 "$COURSE_DATA/permissions.psv" > "$COURSE_OUT/review.psv"
grep -Fx '/opt/collector/bin/report|collector|0775' "$COURSE_OUT/review.psv"
printf 'review_items=1 exploitation_proven=no\n'


## Checks

각 STEP의 test는 고정 자료의 계산 결과를 검사합니다. 아래는 원본 내용 보존을 확인합니다. 실행 성공과 침해 판정은 다릅니다. 어떤 결과가 사실이고 어떤 결론이 가설인지 교안 질문에 답합니다.


In [ ]:
after = {name: hashlib.sha256((data / name).read_bytes()).hexdigest() for name in fixtures}
assert before == after
print('원본 내용 보존: PASS')
print('분석 결과 파일 수:', sum(p.is_file() for p in output.rglob('*')))


## Next Steps

교안의 완료 기준에 따라 근거·정상 행위 가능성·누락·추가 확인을 제출합니다. 결과는 검토용 임시 폴더에 남습니다. 재실행은 Setup부터 새 폴더에서 시작하며 실제 증거를 공개 저장소에 올리지 않습니다.
